#### Setup

# Run 1.Exploration inline here to get all the views

In [0]:
%run "./1.Exploration"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Years window that we are focussing on
YEAR_START = 2015
YEAR_END   = 2025

# Fossil fuels in scope for the narrative
FOSSIL_FUELS = ["coal", "gas", "oil"]

# Minimum generation threshold for including more plants in analysis
MIN_GENERATION_MWH = 50_000


# Final output path
GOLD_DIR = "/dbfs/FileStore/pudl_gold"

print(f"Year window: {YEAR_START}–{YEAR_END}")
print(f"Fossil fuels: {FOSSIL_FUELS}")

#### Views required from Exploration

In [0]:
required_views = [
    "ferc_steam_onm_first",
    "ferc_steam_fuel_first",
    "eia_generators_first",
    "eia_plants_first",
    "ferc_pudl_first",
    "eia_pudl_first",
]

for view in required_views:
    try:
        n = spark.sql(f"SELECT COUNT(*) AS n FROM {view}").collect()
        n = n[0]["n"]
        print(view, "->", n, "rows")
    except Exception:
        print(view, "-> MISSING. Re-run Notebook 1 first.")

#### reducing views to only our interested Working Window(i.e 2015 - 2025)

In [0]:
# FERC tables
ferc_onm_window = spark.sql(f"""
    SELECT *
    FROM ferc_steam_onm_first
    WHERE report_year BETWEEN {YEAR_START} AND {YEAR_END}
""")

ferc_fuel_window = spark.sql(f"""
    SELECT *
    FROM ferc_steam_fuel_first
    WHERE report_year BETWEEN {YEAR_START} AND {YEAR_END}
""")

# EIA tables
# Changing column name to match for both tables
eia_generators_window = spark.sql(f"""
    SELECT *, YEAR(report_date) AS report_year
    FROM eia_generators_first
    WHERE YEAR(report_date) BETWEEN {YEAR_START} AND {YEAR_END}
""")

eia_plants_window = spark.sql(f"""
    SELECT *, YEAR(report_date) AS report_year
    FROM eia_plants_first
    WHERE YEAR(report_date) BETWEEN {YEAR_START} AND {YEAR_END}
""")

# views for futher processing
ferc_onm_window.createOrReplaceTempView("ferc_onm_window")
ferc_fuel_window.createOrReplaceTempView("ferc_fuel_window")
eia_generators_window.createOrReplaceTempView("eia_generators_window")
eia_plants_window.createOrReplaceTempView("eia_plants_window")

print("ferc_onm_window:", ferc_onm_window.count())
print("ferc_fuel_window:", ferc_fuel_window.count())
print("eia_generators_window:", eia_generators_window.count())
print("eia_plants_window:", eia_plants_window.count())

## Operational Cost per Megawatthour
calculating opex per mwh safely by making 0 and NULL results NULL.

In [0]:
ferc_onm_clean = ferc_onm_window.withColumn(
    "opex_per_mwh_safe",
    F.when(
        (F.col("net_generation_mwh") > 0)
        & F.col("opex_production_total").isNotNull(),
        F.col("opex_production_total") / F.col("net_generation_mwh")
    ).otherwise(F.lit(None))
)

ferc_onm_clean.createOrReplaceTempView("ferc_onm_clean")

print("Rows:", ferc_onm_clean.count())
ferc_onm_clean.select("opex_per_mwh_safe").describe().show()

#### flag plants that cost money but produce nothing

In [0]:
ferc_onm_clean = ferc_onm_clean.withColumn(
    "zombie_plant",
    F.when(
        # Coalesce returns the first non-null value of the 2 
        #  Effectively changing Null Valued to 0
        (F.coalesce(F.col("plant_hours_connected_while_generating"), F.lit(0)) == 0)
        & (F.col("opex_production_total") > 0),
        F.lit(True)
    ).otherwise(F.lit(False))
)

ferc_onm_clean.createOrReplaceTempView("ferc_onm_clean")
# Quick check
ferc_onm_clean.groupBy("zombie_plant").count().show()

#### For labelling Plants we pick one primary fuel for 1 plant
A same plant can have multiple fossil fuel source, while the opex and mwh reflects the aggregate of all fuels, it is better for labelling graphs in the next Notebook that we pickup only 1 Fuel name per plant per year.

In [0]:
max_units_per_plant = (
    ferc_fuel_window
    .filter(F.col("fuel_type_code_pudl").isNotNull())
    .groupBy("utility_id_ferc1", "report_year", "plant_name_ferc1")
    .agg(F.max("fuel_consumed_units").alias("max_units"))
)

print("Plant-years with at least one fuel row:", max_units_per_plant.count())


ferc_fuel_primary = (
    ferc_fuel_window
    .filter(F.col("fuel_type_code_pudl").isNotNull())
    .join(
        max_units_per_plant,
        on=["utility_id_ferc1", "report_year", "plant_name_ferc1"],
        how="inner"
    )
    .filter(F.col("fuel_consumed_units") == F.col("max_units"))
    .select(
        "utility_id_ferc1",
        "report_year",
        "plant_name_ferc1",
        F.col("fuel_type_code_pudl").alias("primary_fuel"),
        "fuel_consumed_units",
        "fuel_cost_per_mmbtu",
    )
    # Incase of 2 or more fuels with same consumption.
    .dropDuplicates(["utility_id_ferc1", "report_year", "plant_name_ferc1"])
)


ferc_fuel_primary.createOrReplaceTempView("ferc_fuel_primary")
print("Rows after picking only 1 fuel:", ferc_fuel_primary.count())
ferc_fuel_primary.groupBy("primary_fuel").count().orderBy(F.desc("count")).show()


#### keeping only currently operating fossil-fuel generators in EIA

In [0]:
eia_gen_clean = eia_generators_window.filter(
    (F.col("operational_status") == "existing")
    & (F.col("fuel_type_code_pudl").isin(FOSSIL_FUELS))
)

eia_gen_clean.createOrReplaceTempView("eia_gen_clean")

print("Generators after filter:", eia_gen_clean.count())
eia_gen_clean.groupBy("fuel_type_code_pudl").count().show()

#### Joining generator level data to Plant level Data
Aggregating the produce and Capacity off all the generators in a Plant that are working and adding joining them into the plant table

In [0]:
# Taking Average Capacity and Mwh Power produced by each generator
eia_gen_weighted = (
    eia_gen_clean
    .withColumn("heat_rate_x_capacity",
                F.col("unit_heat_rate_mmbtu_per_mwh") * F.col("capacity_mw"))
    .withColumn("cap_factor_x_capacity",
                F.col("capacity_factor") * F.col("capacity_mw"))
)

eia_gen_weighted.createOrReplaceTempView("eia_gen_weighted")


# Aggregating all generators by plant
eia_plant_agg = spark.sql("""
    SELECT
        plant_id_eia,
        plant_id_pudl,    
        report_year,
        utility_id_pudl,

        FIRST(utility_id_eia) AS utility_id_eia,
        FIRST(utility_name_eia) AS utility_name_eia,
        FIRST(plant_name_eia) AS plant_name_eia,
        FIRST(state) AS state,
        FIRST(county) AS county,

        SUM(capacity_mw) AS plant_capacity_mw,
        SUM(net_generation_mwh) AS plant_net_generation_mwh,
        SUM(total_fuel_cost) AS plant_total_fuel_cost,

        SUM(heat_rate_x_capacity) AS sum_heat_rate_x_capacity,
        SUM(cap_factor_x_capacity) AS sum_cap_factor_x_capacity,

        FIRST(fuel_type_code_pudl) AS eia_primary_fuel,
        FIRST(ownership_code) AS ownership_code,
        COUNT(generator_id) AS generator_count
    FROM eia_gen_weighted
    GROUP BY plant_id_eia, plant_id_pudl,  report_year, utility_id_pudl
""")

eia_plant_agg.createOrReplaceTempView("eia_plant_agg")

print("Generator rows after aggregation:", eia_plant_agg.count())

eia_plant_clean = (
    eia_plant_agg
    .withColumn("plant_heat_rate",
                F.col("sum_heat_rate_x_capacity") / F.col("plant_capacity_mw"))
    .withColumn("plant_capacity_factor",
                F.col("sum_cap_factor_x_capacity") / F.col("plant_capacity_mw"))
)

eia_plant_clean.createOrReplaceTempView("eia_plant_clean")

In [0]:
display(eia_plant_clean.limit(5))

#### Seperating Plans based on their Sector (Private or Public)

Which Sector belongs to which can be seen [here](https://www.eia.gov/electricity/data/guide/pdf/guide.pdf)

In [0]:
plant_sector_clean = eia_plants_window.withColumn(
    "utility_class",
    F.when(F.col("sector_name_eia").isin(["Electric Utility", "NAICS-22 Non-Cogen"]),
           "regulated_utility")
     .when(F.col("sector_name_eia").startswith("IPP"),
           "Independent")
     .when(F.col("sector_name_eia").startswith("Industrial") |
            F.col("sector_name_eia").startswith("Commercial"),
           "self_generation")
     .otherwise("other")
)

plant_sector_clean.createOrReplaceTempView("plant_sector_clean")
plant_sector_clean.groupBy("utility_class").count().orderBy(F.desc("count")).show()

In [0]:
display(plant_sector_clean.limit(5))

### Joining FERC Utility level data with Plant level

In [0]:
ferc_with_pudl = spark.sql("""
    SELECT
        f.*,
        u.utility_id_pudl,
        u.utility_name_ferc1,
        p.plant_id_pudl
    FROM ferc_onm_clean f
    JOIN ferc_pudl_first u
        ON f.utility_id_ferc1 = u.utility_id_ferc1
    JOIN ferc_pudl_plants_first p
        ON f.utility_id_ferc1 = p.utility_id_ferc1
       AND f.plant_name_ferc1 = p.plant_name_ferc1
""")

ferc_with_pudl.createOrReplaceTempView("ferc_with_pudl")
print("FERC rows with PUDL utility + plant IDs attached:", ferc_with_pudl.count())

In [0]:
ferc_eia_joined = spark.sql("""
    SELECT
        fp.*,
        ep.plant_id_eia,
        ep.utility_id_eia,
        ep.utility_name_eia,
        ep.plant_name_eia,
        ep.plant_capacity_mw AS eia_capacity_mw,
        ep.plant_heat_rate,
        ep.plant_capacity_factor,
        ep.eia_primary_fuel,
        ep.state,
        ep.county,
        ep.ownership_code,
        ep.generator_count
    FROM ferc_with_pudl fp
    JOIN eia_plant_clean ep
        ON fp.plant_id_pudl = ep.plant_id_pudl
       AND fp.report_year   = ep.report_year
""")

ferc_eia_joined.createOrReplaceTempView("ferc_eia_joined")
print("Plant-year rows after FERC + EIA join:", ferc_eia_joined.count())
# seeing how similar the names are
ferc_eia_joined.select("plant_name_ferc1", "plant_name_eia").show(10)

## Adding fuel and sector data

In [0]:
ferc_eia_enriched = spark.sql("""
    SELECT
        fe.*,
        ff.primary_fuel          AS ferc_primary_fuel,
        ff.fuel_cost_per_mmbtu,
        ps.sector_name_eia,
        ps.utility_class
    FROM ferc_eia_joined fe
    LEFT JOIN ferc_fuel_primary ff
        ON fe.utility_id_ferc1 = ff.utility_id_ferc1
       AND fe.plant_name_ferc1 = ff.plant_name_ferc1
       AND fe.report_year      = ff.report_year
    LEFT JOIN plant_sector_clean ps
        ON fe.plant_id_eia = ps.plant_id_eia
       AND fe.report_year  = ps.report_year
""")

ferc_eia_enriched.createOrReplaceTempView("ferc_eia_enriched")
print("Plant-year rows after enrichment:", ferc_eia_enriched.count())

In [0]:
display(ferc_eia_enriched.limit(5))

In [0]:
plant_economics_final = spark.sql(f"""
    SELECT
        utility_id_ferc1,
        utility_name_ferc1,
        utility_id_pudl,
        utility_id_eia,
        utility_name_eia,
        plant_id_pudl,
        plant_id_eia,
        plant_name_ferc1,
        plant_name_eia,
        report_year,

        capacity_mw AS ferc_capacity_mw,
        eia_capacity_mw,
        net_generation_mwh,
        plant_capacity_factor,
        plant_heat_rate,

        opex_production_total,
        opex_fuel,
        opex_per_mwh_safe,
        capex_total,
        zombie_plant,

        ferc_primary_fuel,
        eia_primary_fuel,
        fuel_cost_per_mmbtu,

        state,
        county,
        sector_name_eia,
        utility_class,
        ownership_code,

        plant_type AS ferc_plant_type,
        generator_count
    FROM ferc_eia_enriched
   WHERE net_generation_mwh >= {MIN_GENERATION_MWH}
       OR zombie_plant = TRUE
""")

plant_economics_final.createOrReplaceTempView("plant_economics_final")
print("rows:", plant_economics_final.count())
display(plant_economics_final.limit(5))

In [0]:
utility_summary_final = spark.sql("""
    SELECT
        utility_id_ferc1,
        utility_id_pudl,
        report_year,

        -- same for every row of a utility-year
        FIRST(utility_name_ferc1) AS utility_name_ferc1,
        FIRST(utility_name_eia)   AS utility_name_eia,
        FIRST(sector_name_eia)    AS sector_name_eia,
        FIRST(utility_class)      AS utility_class,

        --  Aggregations
        COUNT(DISTINCT plant_id_eia) AS plant_count,
        SUM(net_generation_mwh)      AS total_generation_mwh,
        SUM(opex_production_total)   AS total_opex,
        SUM(opex_fuel)               AS total_fuel_opex,

        -- Generation weighted opex per MWh
        SUM(opex_production_total) / NULLIF(SUM(net_generation_mwh),0) AS weighted_opex_per_mwh,

        AVG(plant_capacity_factor)   AS avg_capacity_factor,
        AVG(plant_heat_rate)         AS avg_heat_rate,

        SUM(CASE WHEN zombie_plant THEN 1 ELSE 0 END) AS zombie_plant_count
    FROM plant_economics_final
    GROUP BY utility_id_ferc1, utility_id_pudl, report_year
""")

utility_summary_final.createOrReplaceTempView("utility_summary_final")
print("Utility-year rows:", utility_summary_final.count())
display(utility_summary_final.limit(5))